In [1]:
import os
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
LABEL_MAP = {0: "A", 1: "B", 2: "C", 3: "D", 4: "E"}

TRAIN_PATH = '/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv'
TEST_PATH = '/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv'
SAMPLE_PATH = '/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv'

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
sample_df = pd.read_csv(SAMPLE_PATH)

DEBERTA_PATH = "microsoft/deberta-v3-small"
ROBERTA_PATH = "roberta-base"

tokenizer_deberta = AutoTokenizer.from_pretrained(DEBERTA_PATH)
model_deberta = AutoModelForSequenceClassification.from_pretrained(DEBERTA_PATH).to(DEVICE).eval()

tokenizer_roberta = AutoTokenizer.from_pretrained(ROBERTA_PATH)
model_roberta = AutoModelForSequenceClassification.from_pretrained(ROBERTA_PATH).to(DEVICE).eval()

def predict_probabilities(model, tokenizer, texts, batch_size=32):
    all_probs = []
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i + batch_size]
        inputs = tokenizer(batch_texts, padding=True, truncation=True, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            logits = model(**inputs).logits
            probs = torch.softmax(logits, dim=-1).cpu().numpy()
            all_probs.append(probs)
    return np.vstack(all_probs)

prompt_col = "prompt" if "prompt" in test_df.columns else "question"
prompts = test_df[prompt_col].tolist()

probs_deberta = predict_probabilities(model_deberta, tokenizer_deberta, prompts)
probs_roberta = predict_probabilities(model_roberta, tokenizer_roberta, prompts)

idx = 25
p_deb_25 = probs_deberta[idx]
top1_deb_idx = np.argmax(p_deb_25)
q1_option = LABEL_MAP[top1_deb_idx]
q1_prob = p_deb_25[top1_deb_idx]

print(f"--- Question 1 ---")
print(f"Answer: {q1_option}, {q1_prob:.4f}\n")

p_avg_25 = (probs_deberta[idx] + probs_roberta[idx]) / 2.0
q2_option = LABEL_MAP[np.argmax(p_avg_25)]

print(f"--- Question 2 ---")
print(f"Answer: {q2_option}\n")

probs_weighted = 0.70 * probs_deberta + 0.30 * probs_roberta
p_w_25 = probs_weighted[idx]

ranked_indices_25 = np.argsort(p_w_25)[::-1]
q3_option = LABEL_MAP[ranked_indices_25[0]]
q4_top3_str = " ".join([LABEL_MAP[i] for i in ranked_indices_25[:3]])

print(f"--- Question 3 ---")
print(f"Answer: {q3_option}\n")

print(f"--- Question 4 ---")
print(f"Answer: {q4_top3_str}\n")

top3_predictions = []
for p in probs_weighted:
    top3_idx = np.argsort(p)[::-1][:3]
    top3_predictions.append(" ".join([LABEL_MAP[i] for i in top3_idx]))

submission_df = pd.DataFrame({
    "id": test_df["id"] if "id" in test_df.columns else test_df.index,
    "prediction": top3_predictions
})

submission_df.to_csv("/kaggle/working/submission.csv", index=False)

print(f"--- Question 5 ---")
print(f"Answer: {len(submission_df)}\n")

first_50_prompts = prompts[:50]
aug_50_prompts = ["Answer the following multiple-choice question carefully: " + p for p in first_50_prompts]

probs_deb_orig_50 = probs_deberta[:50]
probs_deb_aug_50 = predict_probabilities(model_deberta, tokenizer_deberta, aug_50_prompts)

probs_deb_tta_50 = (probs_deb_orig_50 + probs_deb_aug_50) / 2.0

top1_orig_50 = np.argmax(probs_deb_orig_50, axis=1)
top1_tta_50 = np.argmax(probs_deb_tta_50, axis=1)

q6_diff_count = np.sum(top1_orig_50 != top1_tta_50)

print(f"--- Question 6 ---")
print(f"Answer: {q6_diff_count}\n")

deb_100 = probs_deberta[:100]
ens_100 = probs_weighted[:100]

deb_top1_100 = np.argmax(deb_100, axis=1)
ens_top1_100 = np.argmax(ens_100, axis=1)

q7_diff_top1 = np.sum(deb_top1_100 != ens_top1_100)

deb_conf_100 = np.max(deb_100, axis=1)
ens_conf_100 = np.max(ens_100, axis=1)
conf_gain = ens_conf_100 - deb_conf_100
q8_pos_gain = np.sum(conf_gain > 0)

deb_top3_strs = [" ".join([LABEL_MAP[i] for i in np.argsort(p)[::-1][:3]]) for p in deb_100]
ens_top3_strs = [" ".join([LABEL_MAP[i] for i in np.argsort(p)[::-1][:3]]) for p in ens_100]

q9_diff_top3 = np.sum([d != e for d, e in zip(deb_top3_strs, ens_top3_strs)])

print(f"--- Question 7 ---")
print(f"Answer: {q7_diff_top1}\n")

print(f"--- Question 8 ---")
print(f"Answer: {q8_pos_gain}\n")

print(f"--- Question 9 ---")
print(f"Answer: {q9_diff_top3}\n")

def calculate_map3(predictions, targets):
    scores = []
    for pred, target in zip(predictions, targets):
        pred_list = pred.split()
        if target in pred_list:
            rank = pred_list.index(target) + 1
            scores.append(1.0 / rank)
        else:
            scores.append(0.0)
    return np.mean(scores)

if "answer" in test_df.columns:
    targets_100 = test_df["answer"].iloc[:100].tolist()
    q10_map3 = calculate_map3(ens_top3_strs, targets_100)
    print(f"--- Question 10 ---")
    print(f"Answer: {q10_map3:.4f}\n")
elif "answer" in train_df.columns and len(train_df) >= 100:
    train_top3_strs = [" ".join([LABEL_MAP[i] for i in np.argsort(p)[::-1][:3]]) for p in probs_weighted[:100]]
    targets_100 = train_df["answer"].iloc[:100].tolist()
    q10_map3 = calculate_map3(train_top3_strs, targets_100)
    print(f"--- Question 10 ---")
    print(f"Answer: {q10_map3:.4f}\n")

config.json:   0%|          | 0.00/578 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/286M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


--- Question 1 ---
Answer: A, 0.5625

--- Question 2 ---
Answer: A

--- Question 3 ---
Answer: A

--- Question 4 ---
Answer: A B

--- Question 5 ---
Answer: 500

--- Question 6 ---
Answer: 9

--- Question 7 ---
Answer: 24

--- Question 8 ---
Answer: 52

--- Question 9 ---
Answer: 24

--- Question 10 ---
Answer: 0.3600

